# 2장. 토크나이저를 직접 학습시키기

자주 붙어 나오는 글자를 하나의 토큰으로 합치는 바이트 쌍 인코딩(BPE)을 직접 구현하고, tokenizers 라이브러리와 결과를 맞춘 다음, 어휘 크기를 바꿔 가며 1장의 언어 모델을 다시 학습시킨다.

**준비:** NSMC의 `ratings_train.txt`를 왼쪽 파일 창에 올리고, 코랩에서는 `!pip install tokenizers`를 먼저 실행한다. 저자의 맥(CPU 스레드 2개)에서 끝까지 실행하는 데 47분쯤 걸렸다.

## 2.1 BPE를 직접 구현하기

In [ ]:
import math, time, re, sys, platform
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
import os
os.environ["RAYON_NUM_THREADS"] = "2"                # tokenizers도 스레드 2개만 쓰게 한다
import tokenizers
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

torch.set_num_threads(2)
raw = pd.read_csv("ratings_train.txt", sep="\t").dropna(subset=["document"])
raw = raw.reset_index(drop=True)
order = np.random.default_rng(0).permutation(len(raw))   # 1장과 같은 나누기
texts_tr = raw.iloc[order[:30000]]["document"].tolist()
texts_va = raw.iloc[order[-10000:]]["document"].tolist()
texts_un = raw.iloc[order[30000:-10000]]["document"].tolist()

counts = Counter(ch for text in texts_tr for ch in text)
letters = {ch for ch, n in counts.items() if n >= 5}      # 1장의 글자 어휘
UNK_CH, SP = "□", "▁"

def clean(text):
    """1장처럼 64글자로 자르고, 어휘에 없는 글자는 □, 공백은 ▁로 바꾼다."""
    return "".join(ch if ch in letters else UNK_CH for ch in text[:64]).replace(" ", SP)

un = [clean(t) for t in texts_un]            # 언어 모델과 토크나이저를 학습시키는 문장
va = [clean(t) for t in texts_va]            # 검증 문장
dev = [clean(t) for t in texts_tr[:5000]]    # 학습률을 고를 때만 쓰는 문장
symbols = sorted(letters - {" "} | {SP, UNK_CH})
C_va = sum(len(t) + 1 for t in va)           # 글자 수 + 문장마다 <끝> 하나
C_dev = sum(len(t) + 1 for t in dev)
print(len(symbols), len(un), len(va), len(dev), C_va)
print(sys.version.split()[0], torch.__version__, tokenizers.__version__,
      platform.machine(), torch.get_num_threads())

In [ ]:
def words_of(text):
    """▁ 바로 앞에서 나눈다. ▁는 뒤 어절의 첫 글자가 된다."""
    return [w for w in re.split(f"(?={SP})", text) if w]

freq = Counter(w for t in un for w in words_of(t))
print(words_of(va[0]))
print(len(freq), sum(freq.values()))
print(freq.most_common(8))

In [ ]:
def count_pairs(words_freq):
    """어절마다 이웃한 두 토큰의 쌍을 어절의 횟수만큼 센다."""
    c = Counter()
    for w, n in words_freq.items():
        for a, b in zip(w, w[1:]):
            c[a, b] += n
    return c

def merge_once(words_freq, a, b):
    """모든 어절에서 a, b가 이웃한 곳을 a+b 토큰 하나로 합친다."""
    out = {}
    for w, n in words_freq.items():
        joined, k = [], 0
        while k < len(w):
            if k + 1 < len(w) and w[k] == a and w[k + 1] == b:
                joined.append(a + b); k += 2
            else:
                joined.append(w[k]); k += 1
        out[tuple(joined)] = out.get(tuple(joined), 0) + n
    return out

state = {tuple(w): n for w, n in freq.items()}               # 처음에는 글자 하나가 토큰 하나
before = count_pairs(state)
state1 = merge_once(state, ".", ".")
after = count_pairs(state1)
print("병합 전", [("".join(p), n) for p, n in before.most_common(6)])
print("'..' 합친 뒤", [("".join(p) if "".join(p) != "..." else "+".join(p), n) for p, n in after.most_common(6)])
print(len(before), len(after), before[".", "."])
print("합친 뒤 새로 생긴 쌍", [(p, n) for p, n in after.most_common(40) if ".." in p][:4])

In [ ]:
import glob, os
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import FancyBboxPatch
for path in (glob.glob(os.path.expanduser("~/Library/Fonts/NotoSansCJK*.otf"))
             + glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc")):
    font_manager.fontManager.addfont(path)
plt.rcParams.update({"font.family": ["Noto Sans CJK KR", "DejaVu Sans"], "font.size": 9, "axes.unicode_minus": False})
BLUE, ORANGE, GREEN, PURPLE, RED, GRAY = "#2F6FDB", "#F28C28", "#1AA37A", "#8A5CD1", "#E5484D", "#9AA3AF"

fig, axes = plt.subplots(2, 1, figsize=(5.2, 4.5), dpi=260)
label = lambda p: f"{p[0]} + {p[1]}"
for ax, counts, title, color in [(axes[0], before, "① 병합 전: 학습 문장 전체에서 이웃한 두 글자의 횟수(상위 6개)", BLUE),
                                 (axes[1], after, "② 마침표 두 개를 '..' 하나로 합친 뒤 다시 센 횟수(상위 6개)", BLUE)]:
    top = counts.most_common(6)
    names = [label(p) for p, _ in top]
    colors = [ORANGE if k == 0 else (PURPLE if ".." in p else color) for k, (p, _) in enumerate(top)]
    bars = ax.barh(names, [n for _, n in top], color=colors, height=0.6)
    for bar, (p, n) in zip(bars, top):
        ax.text(bar.get_width(), bar.get_y() + bar.get_height() / 2, f"  {n:,}", va="center", fontsize=8, color="#374151")
    ax.invert_yaxis(); ax.set_xlim(0, 86000)
    ax.set_title(title, loc="left", fontsize=8.4, color="#374151")
    ax.spines[["top", "right"]].set_visible(False); ax.tick_params(axis="y", labelsize=8.4)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))
axes[0].text(1500, 0, "가장 많은 쌍 → 첫 번째 병합", va="center", fontsize=8, color="white", fontweight="bold")
axes[1].text(1500, 0, "다음 병합", va="center", fontsize=8, color="white", fontweight="bold")
axes[1].text(84000, 4, "보라: 새 토큰 '..'이 들어간 쌍", ha="right", va="center", fontsize=8, color=PURPLE)
axes[1].set_xlabel("쌍이 나온 횟수(어절의 횟수만큼 더했다)")
fig.tight_layout(); fig.savefig("fig_2_pairs.png"); plt.close(fig)

In [ ]:
def learn_bpe(freq, merges_wanted, symbols, small_first=True):
    """이웃한 두 토큰의 횟수를 어절의 횟수만큼 세고, 가장 많은 쌍을 하나로 합치는 일을
    merges_wanted번 한다. 횟수가 같으면 번호가 작은 쌍을 고른다(small_first=False면 큰 쌍)."""
    names = list(symbols)                                  # 번호 → 토큰 글자
    number = {name: k for k, name in enumerate(names)}
    words = [[number[ch] for ch in w] for w in freq]
    times = list(freq.values())
    pairs, where = Counter(), defaultdict(set)              # 쌍의 횟수, 쌍이 들어 있는 어절
    for i, w in enumerate(words):
        for a, b in zip(w, w[1:]):
            pairs[a, b] += times[i]
            where[a, b].add(i)
    sign = -1 if small_first else 1
    merges, top, ties = [], [], 0
    for _ in range(merges_wanted):
        (a, b), n = max(pairs.items(), key=lambda kv: (kv[1], sign * kv[0][0], sign * kv[0][1]))
        ties += sum(1 for v in pairs.values() if v == n) > 1
        top.append(n)                                       # 합친 쌍의 횟수
        new = len(names)
        names.append(names[a] + names[b])
        merges.append((names[a], names[b]))
        for i in where.pop((a, b)):
            w, t = words[i], times[i]
            for x, y in zip(w, w[1:]):                      # 이 어절의 쌍을 모두 빼고
                pairs[x, y] -= t
            joined, k = [], 0
            while k < len(w):
                if k + 1 < len(w) and w[k] == a and w[k + 1] == b:
                    joined.append(new); k += 2
                else:
                    joined.append(w[k]); k += 1
            words[i] = joined
            for x, y in zip(joined, joined[1:]):            # 합친 뒤의 쌍을 다시 더한다
                pairs[x, y] += t
                where[x, y].add(i)
        del pairs[a, b]
    return names, merges, top, ties

lap = time.perf_counter()
names, merges, top, ties = learn_bpe(freq, 8000 - 3 - len(symbols), symbols)
mine_sec = time.perf_counter() - lap
print(len(merges), len(names) + 3, f"{mine_sec:.0f}초", "동점이 있던 병합", ties)
print(["".join(m) for m in merges[:24]])
print(["".join(m) for m in merges[-8:]])
print("합친 쌍의 횟수: 첫 병합", top[0], "300번째", top[299], "3,000번째", top[2999], "마지막 8개", top[-8:])

In [ ]:
rank = {pair: k for k, pair in enumerate(merges)}

def split_word(word, rank):
    """병합 순위가 가장 앞선 쌍부터 합친다. 합칠 쌍이 없으면 멈춘다."""
    parts = list(word)
    while len(parts) > 1:
        best, k = min((rank.get(pair, math.inf), k) for k, pair in enumerate(zip(parts, parts[1:])))
        if best == math.inf:
            break
        parts[k:k + 2] = [parts[k] + parts[k + 1]]
    return parts

def bpe_tokens(text, rank, memo):
    out = []
    for w in words_of(text):
        if w not in memo:                                   # 같은 어절은 한 번만 나눈다
            memo[w] = split_word(w, rank)
        out += memo[w]
    return out

example = next(t for t in va if 16 <= len(t) <= 20 and t.count(SP) >= 3)   # 공백이 셋 이상인 짧은 문장
for n in [0, 30, 300, 3000, len(merges)]:
    part = {pair: k for pair, k in rank.items() if k < n}
    print(n, bpe_tokens(example, part, {}))

In [ ]:
steps = [0, 30, 300, 3000, len(merges)]
fig, ax = plt.subplots(figsize=(5.2, 3.0), dpi=260)
right = 0
for row, n in enumerate(steps):
    part = {pair: k for pair, k in rank.items() if k < n}
    toks, x, y = bpe_tokens(example, part, {}), 0.0, len(steps) - 1 - row
    for tk in toks:
        w = len(tk) * 0.42 + 0.14
        ax.add_patch(FancyBboxPatch((x, y - 0.3), w, 0.6, boxstyle="round,pad=0,rounding_size=0.08",
                                    fc="#E8F0FD" if len(tk) == 1 else "#FDEBD8", ec=BLUE if len(tk) == 1 else ORANGE, lw=0.8))
        ax.text(x + w / 2, y, tk, ha="center", va="center", fontsize=9, color="#1f2937")
        x += w + 0.06
    right = max(right, x)
    ax.text(-0.25, y, f"병합 {n:,}번\n토큰 {len(toks)}개", ha="right", va="center", fontsize=8.4, color="#374151")
ax.set_xlim(-2.75, right + 0.1); ax.set_ylim(-0.6, len(steps) - 0.4); ax.axis("off")
fig.tight_layout(); fig.savefig("fig_2_1.png"); plt.close(fig)

## 2.2 tokenizers 라이브러리와 맞추기

In [ ]:
def char_bpe(size, texts=un, unk=UNK_CH):
    """글자에서 시작하는 BPE. ▁ 앞에서 어절을 나누는 것까지 learn_bpe와 같게 맞춘다."""
    tok = Tokenizer(models.BPE(unk_token=unk))
    tok.pre_tokenizer = pre_tokenizers.Split(SP, behavior="merged_with_next")
    trainer = trainers.BpeTrainer(vocab_size=size, show_progress=False, initial_alphabet=symbols,
                                  special_tokens=["<패딩>", "<시작>", "<끝>"])
    tok.train_from_iterator(texts, trainer)
    return tok

lap = time.perf_counter()
lib = char_bpe(8000)
lib_sec = time.perf_counter() - lap
import json
theirs = [tuple(m) for m in json.loads(lib.to_str())["model"]["merges"]]
same = next((k for k, (p, q) in enumerate(zip(merges, theirs)) if p != q), min(len(merges), len(theirs)))
print(len(theirs), lib.get_vocab_size(), f"{lib_sec:.1f}초", "앞에서부터 같은 병합", same)
memo = {}
mine = [bpe_tokens(t, rank, memo) for t in va]
lib_out = [e.tokens for e in lib.encode_batch(va)]
print("검증 문장 10,000개 가운데 토큰이 모두 같은 문장", sum(a == b for a, b in zip(mine, lib_out)))
names_big, merges_big, *_ = learn_bpe(freq, 300, symbols, small_first=False)
print("동점에서 큰 번호를 고르면 처음 달라지는 병합(몇 번째)", 1 + next(k for k, (p, q) in enumerate(zip(merges, merges_big)) if p != q))

In [ ]:
plain = Tokenizer(models.BPE())                              # unk_token을 주지 않은 BPE
plain.pre_tokenizer = pre_tokenizers.Split(SP, behavior="merged_with_next")
plain.train_from_iterator(un, trainers.BpeTrainer(vocab_size=8000, show_progress=False,
                                                    special_tokens=["<패딩>", "<시작>", "<끝>"]))
raw_va = [t[:64].replace(" ", SP) for t in texts_va]         # □로 바꾸지 않은 검증 문장
lost = [len(t) - len("".join(plain.encode(t).tokens)) for t in raw_va]
print("글자가 사라진 문장", sum(n > 0 for n in lost), "사라진 글자", sum(lost))
print(plain.encode("똠얌꿍▁최고").tokens, lib.encode("똠얌꿍▁최고").tokens)

## 2.3 어휘 크기에 따른 토큰 수

In [ ]:
SIZES = [1424, 2000, 4000, 8000, 16000, 32000]
toks, seen = {}, {}
print("어휘     병합   문장당 토큰  토큰당 글자  학습 자료에 10번 미만 나온 토큰(개, %)")
for size in SIZES:
    tok = char_bpe(size)
    toks[size] = tok
    n_va = sum(len(e.ids) for e in tok.encode_batch(va))
    used = Counter(i for e in tok.encode_batch(un) for i in e.ids)
    rare = sum(used[i] < 10 for i in range(3, tok.get_vocab_size()))
    seen[size] = (n_va, rare / (tok.get_vocab_size() - 3) * 100)
    print(f"{size:>6,} {size - 3 - len(symbols):>6,}  {n_va / len(va):>10.2f}  {(C_va - len(va)) / n_va:>10.3f}"
          f"  {rare:>8,}  {seen[size][1]:>5.1f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(5.2, 4.6), dpi=260, sharex=True)
xs = SIZES
ax = axes[0]
ax.plot(xs, [seen[s][0] / len(va) for s in xs], color=BLUE, marker="o", ms=4, lw=1.4)
for s in xs:                                              # 2,000은 가파른 선과 겹치지 않게 오른쪽 위에 적는다
    ax.annotate(f"{seen[s][0] / len(va):.1f}", (s, seen[s][0] / len(va)), xytext=(5, 4) if s == 2000 else (0, 7),
                textcoords="offset points", ha="left" if s == 2000 else "center", fontsize=8, color=BLUE)
ax.set_ylabel("검증 문장 하나의 토큰 수")
ax.set_ylim(0, 36)
ax.set_title("① 어휘가 커질수록 문장 하나의 토큰은 줄어든다", fontsize=8.6, loc="left", color="#374151")
ax = axes[1]
ax.bar(xs, [seen[s][1] for s in xs], width=[s * 0.25 for s in xs], color=[GRAY] * 5 + [ORANGE])
for s in xs:
    ax.annotate(f"{seen[s][1]:.1f}%", (s, seen[s][1]), xytext=(0, 3), textcoords="offset points", ha="center", fontsize=8)
ax.set_ylabel("학습 문장에 10번 미만\n나온 토큰의 비율(%)", linespacing=1.5)
ax.set_ylim(0, 42)
ax.set_title("② 어휘가 너무 크면 학습 문장에 거의 나오지 않는 토큰이 늘어난다", fontsize=8.6, loc="left", color="#374151")
ax.set_xscale("log")
ax.set_xticks(xs); ax.set_xticklabels([f"{s:,}" for s in xs]); ax.minorticks_off()
ax.set_xlabel("어휘 크기(특수 토큰 3개 포함, 로그 눈금)")
for ax in axes:
    ax.grid(axis="y", color="#e5e7eb", lw=0.6); ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig("fig_2_2.png"); plt.close(fig)

## 2.4 어휘 크기를 바꿔 언어 모델 학습시키기

In [ ]:
class SelfAttention(nn.Module):
    """1장과 같은 셀프 어텐션. x: (배치, 길이, dim)."""
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.to_q, self.to_k, self.to_v = (nn.Linear(dim, dim) for _ in range(3))
        self.out = nn.Linear(dim, dim)

    def forward(self, x, hide):
        B, L, D = x.shape
        H = self.heads
        split = lambda t: t.view(B, L, H, D // H).transpose(1, 2)
        q, k, v = split(self.to_q(x)), split(self.to_k(x)), split(self.to_v(x))
        scores = q @ k.transpose(-2, -1) / math.sqrt(D // H)
        scores = scores.masked_fill(hide[:, None, None, :], float("-inf"))
        later = torch.triu(torch.ones(L, L, dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(later, float("-inf"))
        return self.out((torch.softmax(scores, dim=-1) @ v).transpose(1, 2).reshape(B, L, D))

class Block(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.attend = SelfAttention(dim, heads)
        self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))

    def forward(self, x, hide):
        x = x + self.attend(self.norm1(x), hide)
        return x + self.mlp(self.norm2(x))

class LanguageModel(nn.Module):
    """1장의 모델에서 어휘 크기(size)만 바꿀 수 있게 했다."""
    def __init__(self, size, dim=64, heads=4, layers=2, length=66):
        super().__init__()
        self.table = nn.Embedding(size, dim)
        self.where = nn.Embedding(length, dim)
        self.blocks = nn.ModuleList(Block(dim, heads) for _ in range(layers))
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, size)

    def forward(self, ids):
        x = self.table(ids) + self.where(torch.arange(ids.shape[1]))
        hide = ids == PAD
        for block in self.blocks:
            x = block(x, hide)
        return self.head(self.norm(x))

PAD, START, END = 0, 1, 2
count = lambda net: sum(p.numel() for p in net.parameters())
for size in [1424, 4000, 16000]:
    assert [toks[size].token_to_id(t) for t in ["<패딩>", "<시작>", "<끝>"]] == [PAD, START, END]
    torch.manual_seed(0)
    net = LanguageModel(size)
    print(size, count(net), count(net.table) + count(net.head))

In [ ]:
def to_tensor(tok, texts):
    """<시작> + 토큰 번호 + <끝>. 남는 칸은 0(패딩)이다."""
    rows = [[START] + e.ids + [END] for e in tok.encode_batch(texts)]
    X = torch.zeros(len(rows), max(map(len, rows)), dtype=torch.long)
    for i, r in enumerate(rows):
        X[i, :len(r)] = torch.tensor(r)
    return X

def batches(lengths, seed, size=64, chunk=50):
    """길이가 비슷한 문장끼리 배치를 만든다(1장과 같다)."""
    shuffler = torch.Generator().manual_seed(seed)
    mixed = torch.randperm(len(lengths), generator=shuffler)
    made = []
    for start in range(0, len(mixed), size * chunk):
        part = mixed[start:start + size * chunk]
        part = part[torch.argsort(lengths[part])]
        made += [part[k:k + size] for k in range(0, len(part), size)]
    return [made[k] for k in torch.randperm(len(made), generator=shuffler)]

def loss_sum(net, data):
    """목표 토큰의 손실 합(자연로그)과 목표 토큰 수."""
    net.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for k in range(0, len(data), 500):
            ids = data[k:k + 500]
            out = net(ids[:, :-1])
            total += F.cross_entropy(out.reshape(-1, out.shape[-1]), ids[:, 1:].reshape(-1),
                                     ignore_index=PAD, reduction="sum").item()
            n += (ids[:, 1:] != PAD).sum().item()
    return total, n

def fit(net, X, lr, seed, epochs):
    """1장의 fit과 같다. AdamW, 배치 64, 처음 10% 걸음 동안 학습률을 올리고 코사인으로 줄인다."""
    optimizer = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=0.0)
    lengths = (X != PAD).sum(dim=1)
    total, step = epochs * math.ceil(len(X) / 64), 0
    warm = max(1, total // 10)
    for epoch in range(epochs):
        net.train()
        for rows in batches(lengths, seed * 1000 + epoch):
            step += 1
            wave = 0.5 * (1 + math.cos(math.pi * step / total))
            for group in optimizer.param_groups:
                group["lr"] = lr * min(1, step / warm) * wave
            ids = X[rows][:, :int(lengths[rows].max())]
            optimizer.zero_grad()
            out = net(ids[:, :-1])
            F.cross_entropy(out.reshape(-1, out.shape[-1]), ids[:, 1:].reshape(-1),
                            ignore_index=PAD).backward()
            optimizer.step()
    return net

def bpc(net, data, chars):
    total, n = loss_sum(net, data)
    return total / math.log(2) / chars, total / n          # 글자당 비트, 토큰 하나당 손실

def run(size, lr, seed, tok_of=None, epochs=4):
    tok_of = tok_of or toks
    X_un, X_dev, X_va = data[size]
    torch.manual_seed(seed)
    lap = time.perf_counter()
    length = max([66] + [X.shape[1] for X in data[size]])       # 글자 BPE는 66칸(1장과 같다)
    net = fit(LanguageModel(tok_of[size].get_vocab_size(), length=length), X_un, lr, seed, epochs)
    sec = time.perf_counter() - lap
    nets[size, lr, seed] = net                                  # 2.5절에서 다시 쓴다
    return bpc(net, X_dev, C_dev)[0], *bpc(net, X_va, C_va), sec

In [ ]:
LM_SIZES, RATES = [1424, 2000, 4000, 8000, 16000], [0.003, 0.01, 0.03]
data, runs, nets = {}, {}, {}
print("어휘     학습률   개발 BPC   시간(4에폭)")
for size in LM_SIZES:
    data[size] = [to_tensor(toks[size], t) for t in (un, dev, va)]
    for lr in RATES:
        runs[size, lr, 0] = run(size, lr, 0)
        print(f"{size:>6,}  {lr:>6}  {runs[size, lr, 0][0]:>8.4f}  {runs[size, lr, 0][3]:>7.0f}초", flush=True)

In [ ]:
best = {size: min(RATES, key=lambda lr: runs[size, lr, 0][0]) for size in LM_SIZES}
for size in LM_SIZES:
    for seed in [1, 2]:
        runs[size, best[size], seed] = run(size, best[size], seed)
print("어휘     고른 학습률  검증 BPC(시드 3개 중앙값, 범위)     토큰당 손실  1에폭 시간   학습률 0.03의 BPC")
final = {}
for size in LM_SIZES:
    v = [runs[size, best[size], s][1] for s in range(3)]
    final[size] = float(np.median(v))
    loss = np.median([runs[size, best[size], s][2] for s in range(3)])
    sec = np.median([runs[size, best[size], s][3] for s in range(3)]) / 4
    print(f"{size:>6,}  {best[size]:>9}  {final[size]:>8.4f} ({min(v):.4f}~{max(v):.4f})  {loss:>12.4f}"
          f"  {sec:>8.1f}초  {runs[size, 0.03, 0][1]:>10.4f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(5.2, 5.8), dpi=260, sharex=True)
ax = axes[0]
for lr, color, style in [(0.003, GRAY, ":"), (0.01, GREEN, "-"), (0.03, ORANGE, "--")]:
    ys = [runs[s, lr, 0][0] for s in LM_SIZES]
    shown = [min(y, 4.47) for y in ys]                    # 칸을 넘는 값은 위 끝에 세모로 적는다
    ax.plot(LM_SIZES, shown, color=color, ls=style, lw=1.4, label=f"학습률 {lr}")
    for s, y, v in zip(LM_SIZES, shown, ys):
        ax.plot([s], [y], marker="^" if v > 4.47 else "o", ms=5 if v > 4.47 else 3.6, color=color)
        if v > 4.47:
            ax.annotate(f"{v:.4f}\n(칸 밖)", (s, y), xytext=(0, -9), textcoords="offset points", ha="center",
                        va="top", fontsize=8, color=color, bbox=dict(fc="white", ec="none", pad=0.6))
ax.set_ylim(4.12, 4.50)
ax.set_ylabel("개발 문장의 글자당 비트")
ax.set_title("① 시드 0, 4에폭. 어휘가 커지면 잘 맞는 학습률이 내려간다", fontsize=8.6, loc="left", color="#374151")
ax.legend(frameon=False, fontsize=8, ncol=3, loc="lower left", columnspacing=1.2, handlelength=1.8)
ax = axes[1]
med = [final[s] for s in LM_SIZES]
lo = [med[k] - min(runs[s, best[s], r][1] for r in range(3)) for k, s in enumerate(LM_SIZES)]
hi = [max(runs[s, best[s], r][1] for r in range(3)) - med[k] for k, s in enumerate(LM_SIZES)]
ax.errorbar(LM_SIZES, med, yerr=[lo, hi], color=BLUE, marker="o", ms=4, lw=1.4, capsize=3)
for k, s in enumerate(LM_SIZES):
    sec = np.median([runs[s, best[s], r][3] for r in range(3)]) / 4
    up = k % 2 == 0                                       # 이웃한 어휘의 글자가 겹치지 않게 막대 위와 아래에 번갈아 적는다
    ax.annotate(f"{med[k]:.3f}\n학습률 {best[s]}\n1에폭 {sec:.0f}초", (s, med[k] + hi[k] if up else med[k] - lo[k]),
                xytext=(0, 6 if up else -6), textcoords="offset points", ha="center", va="bottom" if up else "top",
                fontsize=8, color="#374151")
ax.set_ylim(4.08, 4.30); ax.yaxis.set_major_locator(plt.MultipleLocator(0.05))
ax.set_ylabel("검증 문장의 글자당 비트")
ax.set_title("② 어휘마다 고른 학습률로 시드 3개(점은 중앙값, 막대는 범위)", fontsize=8.6, loc="left", color="#374151")
ax.set_xscale("log"); ax.set_xlim(1100, 18500)
ax.set_xticks(LM_SIZES); ax.set_xticklabels([f"{s:,}" for s in LM_SIZES]); ax.minorticks_off()
ax.set_xlabel("어휘 크기(로그 눈금)")
for ax in axes:
    ax.grid(axis="y", color="#e5e7eb", lw=0.6); ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig("fig_2_3.png"); plt.close(fig)

## 2.5 바이트에서 시작하는 BPE

In [ ]:
def byte_bpe(size, texts=texts_un):
    """UTF-8 바이트 256개에서 시작하는 BPE. 모르는 글자가 없다."""
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.Sequence([
        pre_tokenizers.Split(" ", behavior="merged_with_next"),
        pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False)])
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=size, show_progress=False,
                                  initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
                                  special_tokens=["<패딩>", "<시작>", "<끝>"])
    tok.train_from_iterator([t[:64] for t in texts], trainer)
    return tok

print("한".encode("utf-8"), len("영화 정말 재밌다".encode("utf-8")), len("영화 정말 재밌다"))
raw64 = [t[:64] for t in texts_va]
btoks, B, broken = {}, {}, {}
print("어휘     문장당 토큰(바이트 BPE)  글자 BPE와의 비율  온전한 글자가 아닌 토큰")
for size in [259] + SIZES[1:]:
    tok = byte_bpe(size)
    btoks[size] = tok
    n = B[size] = sum(len(e.ids) for e in tok.encode_batch(raw64))
    broken[size] = sum("\ufffd" in tok.decode([i]) for i in range(3, tok.get_vocab_size()))
    ref = seen[size][0] if size in seen else seen[1424][0]
    print(f"{size:>6,}  {n / len(va):>18.2f}  {n / ref:>14.3f}  {broken[size]:>14,}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(5.2, 4.6), dpi=260, sharex=True)
xs = SIZES[1:]
ax = axes[0]
for name, ys, color, style in [("글자 BPE", [seen[s][0] / len(va) for s in xs], BLUE, "-"),
                               ("바이트 BPE", [B[s] / len(va) for s in xs], ORANGE, "--")]:
    ax.plot(xs, ys, color=color, ls=style, marker="o", ms=4, lw=1.4, label=name)
for s in xs:
    ax.annotate(f"×{B[s] / seen[s][0]:.3f}", (s, B[s] / len(va)), xytext=(0, 8), textcoords="offset points",
                ha="center", fontsize=8, color=ORANGE)
ax.set_ylabel("검증 문장 하나의 토큰 수")
ax.set_ylim(8, 24)
ax.legend(frameon=False, fontsize=8)
ax.set_title(f"① 바이트 BPE / 글자 BPE의 토큰 수 비(병합이 없을 때는 ×{B[259] / seen[1424][0]:.2f})",
             fontsize=8.6, loc="left", color="#374151")
ax = axes[1]
ax.bar(xs, [broken[s] for s in xs], width=[s * 0.25 for s in xs], color=GRAY)
for s in xs:
    ax.annotate(f"{broken[s]:,}개\n({broken[s] / (s - 3) * 100:.1f}%)", (s, broken[s]), xytext=(0, 3),
                textcoords="offset points", ha="center", fontsize=8)
ax.set_ylabel("온전한 글자가 아닌\n토큰의 수", linespacing=1.5)
ax.set_ylim(0, 1500)
ax.set_title("② 바이트 BPE의 어휘 가운데 글자로 온전히 바뀌지 않는 토큰", fontsize=8.6, loc="left", color="#374151")
ax.set_xscale("log")
ax.set_xticks(xs); ax.set_xticklabels([f"{s:,}" for s in xs]); ax.minorticks_off()
ax.set_xlabel("어휘 크기(특수 토큰 3개 포함, 로그 눈금)")
for ax in axes:
    ax.grid(axis="y", color="#e5e7eb", lw=0.6); ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig("fig_2_4.png"); plt.close(fig)

In [ ]:
def pieces_of(tok, text):
    """토큰마다 (보이는 글자, 온전한 글자인지)를 돌려준다."""
    out = []
    for i in tok.encode(text).ids:
        piece = tok.decode([i])
        out.append((piece, "\ufffd" not in piece))
    return out

b8, c8 = btoks[8000], toks[8000]
pick = next(t for t in raw64 if 8 <= len(t) <= 22 and not all(ok for _, ok in pieces_of(b8, t)))
print(pick)
print("글자 BPE  ", c8.encode(clean(pick)).tokens)
print("바이트 BPE", [p if ok else "(조각)" for p, ok in pieces_of(b8, pick)])
ids = b8.encode(pick).ids
print([b8.decode(ids[:k]) for k in range(1, len(ids) + 1)])
odd = b8.encode("똠얌꿍 최고")
print(len(odd.ids), b8.decode(odd.ids), [p if ok else "(조각)" for p, ok in pieces_of(b8, "똠얌꿍 최고")])

In [ ]:
def bytes_to_unicode():
    """GPT-2가 바이트 256개를 보이는 글자 256개에 하나씩 대응시킨 표. ByteLevel이 쓰는 것과 같은지 확인한다."""
    keep = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
    cs, n = keep[:], 0
    for b in range(256):
        if b not in keep:
            keep.append(b); cs.append(256 + n); n += 1
    return dict(zip(keep, map(chr, cs)))

shown = bytes_to_unicode()
assert set(shown.values()) == set(pre_tokenizers.ByteLevel.alphabet())
back = {ch: b for b, ch in shown.items()}
print(repr(shown[ord(" ")]), [hex(b) for b in "혓".encode("utf-8")])
pieces = []
for i in b8.encode(pick).ids:
    got = bytes(back[ch] for ch in b8.id_to_token(i))           # 토큰 글자를 바이트로 되돌린다
    pieces.append((got, b8.decode([i])))
print([(r.hex(" "), t) for r, t in pieces])
print([(ch, ch.encode("utf-8").hex(" ")) for ch in pick])

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 2.85), dpi=260)
tail = pick[pick.index("묻") - 1:]                          # ' 묻혓다.' 부분만 크게 그린다
start = sum(len(r) for r, _ in pieces) - len(tail.encode("utf-8"))
X0, W = 0.0, 0.42
x, pos, used = X0, 0, 0
cells_x = []
for r, t in pieces:                                           # 토큰마다 그 토큰이 담은 바이트를 칸으로 그린다
    for b in r:
        if pos >= start:
            cells_x.append((x, b))
            x += W
        pos += 1
for k, ch in enumerate(tail):                                  # 맨 위 줄: 글자와 그 글자의 바이트 범위
    nb = len(ch.encode("utf-8"))
    xs = cells_x[used][0]; xe = cells_x[used + nb - 1][0] + W
    ax.add_patch(plt.Rectangle((xs + 0.02, 3.22), xe - xs - 0.04, 0.56, fc="#EEF3FC", ec=BLUE, lw=0.8))
    ax.text((xs + xe) / 2, 3.5, "␣" if ch == " " else ch, ha="center", va="center", fontsize=13, color="#1f2937")
    used += nb
for xb, b in cells_x:                                          # 둘째 줄: 바이트 16진수
    ax.add_patch(plt.Rectangle((xb + 0.02, 2.28), W - 0.04, 0.5, fc="white", ec="#cbd5e1", lw=0.7))
    ax.text(xb + W / 2, 2.53, f"{b:02x}", ha="center", va="center", fontsize=9, family="monospace", color="#374151")
x, pos, shown_text = X0, 0, ""
for j, (r, t) in enumerate(pieces):                            # 셋째 줄: 토큰이 묶은 바이트, 넷째 줄: 그 토큰까지 화면에 보이는 글자
    shown_text = b8.decode(b8.encode(pick).ids[:j + 1])
    n_here = sum(1 for k in range(len(r)) if pos + k >= start)
    if n_here:
        ok = "�" not in t
        ax.add_patch(plt.Rectangle((x + 0.015, 1.3), n_here * W - 0.03, 0.56, fc="#E3F5EE" if ok else "#FDE7E8",
                                   ec=GREEN if ok else RED, lw=1.0))
        ax.text(x + n_here * W / 2, 1.58, t.replace(" ", "␣") if ok else f"조각\n{len(r)}바이트", ha="center", va="center",
                fontsize=9.4 if ok else 8, linespacing=1.1, color="#0F7A57" if ok else "#C53030", fontweight="bold")
        ax.text(x + n_here * W / 2, 0.62, "…" + shown_text[-3:].replace(" ", "␣"), ha="center", va="center", fontsize=9.4,
                color=RED if shown_text.endswith("�") else "#1f2937")
        x += n_here * W
    pos += len(r)
for y, name in [(3.95, "글자"), (2.95, "UTF-8 바이트"), (2.03, "바이트 BPE 토큰"), (1.05, "이 토큰까지 화면에 보이는 끝")]:
    ax.text(X0 + 0.02, y, name, ha="left", va="center", fontsize=8.6, color="#374151", fontweight="bold")
ax.set_xlim(X0 - 0.03, x + 0.03); ax.set_ylim(0.4, 4.12); ax.axis("off")
fig.subplots_adjust(left=0.01, right=0.99, bottom=0.01, top=0.99); fig.savefig("fig_2_bytes.png"); plt.close(fig)

In [ ]:
key = "바이트 4,000"
data[key] = [to_tensor(btoks[4000], t) for t in ([t[:64] for t in texts_un], [t[:64] for t in texts_tr[:5000]], raw64)]
print("가장 긴 문장의 토큰 수", data[key][0].shape[1] - 2, data[key][2].shape[1] - 2)
byte_run = run(key, best[4000], 0, tok_of={key: btoks[4000]})
print(f"바이트 BPE 4,000: 검증 BPC {byte_run[1]:.4f}, 토큰당 손실 {byte_run[2]:.4f}, {byte_run[3]:.0f}초")
print(f"글자 BPE 4,000(같은 학습률, 시드 0): 검증 BPC {runs[4000, best[4000], 0][1]:.4f}")
rows = [i for i, t in enumerate(va) if UNK_CH not in t]                 # 모르는 글자가 없는 검증 문장
chars = sum(len(va[i]) + 1 for i in rows)
for name, net, X in [("글자 BPE 4,000", nets[4000, best[4000], 0], data[4000][2]),
                     ("바이트 BPE 4,000", nets[key, best[4000], 0], data[key][2])]:
    print(f"{name}: 모르는 글자가 없는 문장 {len(rows):,}개에서 BPC {bpc(net, X[rows], chars)[0]:.4f}")